# Tutorial 17: Skyrmion-lattice rods, sample tilt, and Ewald-sphere mapping

**Question:** can a small-period skyrmion lattice scatter weakly at normal incidence, yet show strong diffraction when tilted?

We first predict the answer using an independent first-Born reference, then run the repository's scalar multislice propagator through a rotated three-dimensional texture. Finally we assemble the signals in sample-frame momentum space. All simulation functions are readable in [`experiments.py`](../paper/scattering_calculator/experiments.py); the manuscript and figure notebooks live in [`paper/scattering_calculator`](../paper/scattering_calculator/README.md).

A finite film gives rods with a longitudinal **sinc-squared envelope**, not rods of compact support. We arrange for the Ewald sphere to miss the main lobe at all six first-order rod centres. Residual scattering remains because illumination has finite extent, reciprocal peaks have finite width, and the rods have side lobes. These are magnetic superlattice rods, not atomic crystallographic Bragg peaks.

Runtime is typically a few minutes on a CPU; peak arrays are a few hundred MB. No GPU, downloaded data, or external micromagnetic file is required. Run from the repository, `tutorials`, or paper notebook folder. For dependencies see the paper README. Generated PNG/PDF figures and NPZ data go to `paper/scattering_calculator/results`.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/scattering_calculator").is_dir())
PAPER = ROOT / "paper/scattering_calculator"
sys.path.insert(0, str(PAPER))
import experiments as ex
OUT = PAPER / "results"
OUT.mkdir(exist_ok=True)

def show(name):
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.imshow(plt.imread(OUT / (name + ".png")))
    ax.axis("off")
    plt.show()


## Editable experiment parameters

Change this cell and run the cells below it. Numerical values in the explanatory prose describe the original default design; the printed geometry and saved configuration are authoritative for your run. `thickness_nm=None` retunes the thickness when energy or lattice spacing changes; a number keeps it fixed. Tilt is about lab y and is measured from normal incidence. Detector centre offsets are `(x,y)` in pixels.

In [ ]:
# EDIT HERE, then Run All. These values feed every calculation below.
# Radius is where the texture reaches the uniform background, not lattice spacing.
cfg = ex.SkyrmionConfig(
    energy_eV=778.0,
    lattice_nm=12.0,
    radius_nm=3.4,
    thickness_nm=None,  # None: second-zero design; or set e.g. 100.0 / 269.48 / 400.0
    angles_deg=(-8.7947589, -4.3973795, 0.0, 4.3973795, 8.7947589),
    scan_angles_deg=tuple(np.linspace(-12, 12, 49)),  # Born + matched 3D FFT scan
    volume_angles_deg=tuple(np.linspace(-12, 12, 25)),  # production multislice scan
    detector_n=193,
    detector_pitch_m=13.5e-6,
    detector_distance_m=0.007,
    detector_center_offset_xy_px=(0.0, 0.0),
    beam_sigma_nm=24.0,  # amplitude Gaussian sigma
    born_n=256, born_dx_nm=0.75,
    multislice_n=192, multislice_dx_nm=0.75, multislice_dz_nm=2.0,
    volume_n=128,  # faster real-space grid for the multislice tilt scan
    q_bins=101, q_limit_rad_nm=0.85,
    fft_nz=512, fft_dz_nm=2.0,  # 3D FFT z box = fft_nz * fft_dz_nm; include vacuum
    contrast_channel='xmcd',  # 'mz': scalar control, no tilt-dependent projection
    include_cobalt=True,
)
# Optional: move selected image angles with the analytic Bragg condition.
# from dataclasses import replace
# tb = cfg.geometry()['bragg_deg']
# cfg = replace(cfg, angles_deg=(-2*tb, -tb, 0., tb, 2*tb))

OUT = PAPER / 'results' / 'notebook_skyrmion'  # use a different name for each experiment
OUT.mkdir(parents=True, exist_ok=True)
p = cfg.geometry()
print(p)
print('Ewald sphere misses central rod lobe:', p['outside_central_lobe'])
print('Untilting thickness factor:', p['normal_thickness_factor'])


## 1. Choose energy, small skyrmions, and a thick film

Lengths in the experiment helper are nm; the production propagator receives metres. Momentum is always **angular wavevector in rad/nm**, not cycles/nm. The lab incident beam is along $+z$. An active rotation $R_y(\theta)$ takes sample coordinates to lab coordinates; $\theta=0$ means normal incidence (90° measured from the sample surface).

For lattice parameter $a$, the first triangular-lattice shell is $G=4\pi/(\sqrt{3}a)$. With $k=2\pi/\lambda$ the normal-incidence Ewald mismatch is

$$\Delta q_z=k-\sqrt{k^2-G^2}.$$

A uniform tube of thickness $t$ has intensity factor $\mathrm{sinc}^2(q_z t/2)$, where this written sinc means $\sin(x)/x$. NumPy uses $\sin(\pi x)/(\pi x)$, so the code uses `np.sinc(qz*t/(2*np.pi))**2`.

We set $t=4\pi/\Delta q_z$: the rod centres lie at the **second zero**, outside the central lobe $|q_z|<2\pi/t$. At 778 eV and $a=12$ nm this gives $t\simeq269.48$ nm. The prescribed core-to-background radius is 3.4 nm (full texture diameter 6.8 nm; lattice spacing is not diameter). The $+G_x$ rod crosses $q_z=0$ at $\theta_B=\arcsin(G/2k)\simeq4.40^\circ$.

This texture is a synthetic test, not a claim that such thin tubes are a stable equilibrium in a 269 nm cobalt film. Stability requires an independent micromagnetic model.


In [ ]:
print(p)

## 2. First-Born reference: compare identical intensity scales

We generate a unit-length Néel texture on a triangular lattice, constant along the sample normal. The Fourier-transformed contrast is $\mathbf m-\hat z$, illuminated by a Gaussian with amplitude standard deviation 24 nm. The benchmark uses the scalar XMCD channel $\mathbf m_{lab}\cdot\hat k_{in}$. It is not a complete polarization-resolved resonant cross section.

For each physical flat-detector pixel $(X,Y,L)$, we calculate

$$\mathbf q_{lab}=k\left[\frac{(X,Y,L)}{\sqrt{X^2+Y^2+L^2}}-(0,0,1)\right],\qquad \mathbf q_s=R_y(\theta)^T\mathbf q_{lab}.$$

The detector has 193×193 pixels, 13.5 µm pitch, and distance 7 mm. Its acceptance includes the first shell. The Born amplitude interpolates the **complex** transverse Fourier amplitude at $(q_{sx},q_{sy})$ and multiplies it by $t\,\mathrm{sinc}(q_{sz}t/2)$. We evaluate the Ewald surface during the forward calculation; simply painting a 2D projected-object FFT onto a sphere would not recover thickness interference.

Use the common logarithmic scale to compare signals. At a single tilt, the detector shows peaks where the sphere cuts rods; a complete rod is generally not visible as a line in one frame. The thick-film rocking curve narrows compared with the 20 nm control.


In [ ]:
ex.provenance(OUT, {'notebook': '17_skyrmion', 'design': p})
ex.skyrmion_born(OUT, config=cfg)
show('fig01_geometry')
show('fig02_skyrmion_tilts')

## 3. Assemble a measured 3D reciprocal-space volume

A 49-angle scan from −12° to +12° rotates every Ewald sample into the same sample frame. The helper calculates a pixel signal proportional to $I\,d\Omega$ and divides out $d\Omega=p^2L/r^3$ before gridding. Real counts additionally need incident-flux, exposure, efficiency and polarization corrections.

For each voxel we save **sum/intensity count**, plus a separate coverage array. Empty voxels remain NaN. Repeated observations are averaged, not added as if more coverage meant stronger scattering. This is an intensity-volume assembly, **not phase retrieval or magnetization tomography**. A single rotation axis and limited angles leave missing data. Polarization projection changes with tilt; therefore the assembled scalar-XMCD intensities are not automatically samples of one orientation-independent $|\widetilde m|^2$.

The longitudinal central lobes are short for a thick sample. The 3D views and longitudinal profile show their locations and side lobes; they should not be interpreted as infinitely long, uniformly bright crystallographic rods.


In [ ]:
show('fig03_reciprocal_space')
volume = np.load(OUT / 'reciprocal_volume.npz')
print('Measured voxel fraction:', np.mean(volume['coverage'] > 0))
print('Unmeasured voxels remain NaN:', np.isnan(volume['intensity'][volume['coverage'] == 0]).all())

## 4. Test the production multislice kernel

Now the actual `scalar_wavefronts` propagator sees a rotated volume. Coordinates and magnetization vectors are both rotated; planar interfaces use fractional voxel occupancy. A Gaussian with amplitude width 24 nm suppresses periodic-boundary illumination. At each slice the local complex transmission is followed by the repository's angular-spectrum propagation. Defaults: 192×192 pixels, 0.75 nm pitch, 2 nm longitudinal slices.

Two material controls use the same geometry:

- **Weak synthetic medium:** $n_0=1$, $n_c=10^{-6}$, no absorption. This isolates thickness interference.
- **Tabulated Co:** charge and circular contrast loaded at 778 eV without rescaling. This includes refraction, absorption and stronger interaction within the scalar approximation.

For clarity these panels show $|\mathcal F(E_{texture}-E_{saturated})|^2$, a **computed coherent-field difference**. It is not an experimentally measured helicity difference, and is not the total detector intensity. The FTH notebook instead subtracts measured-style CR and CL intensities.

The repository uses `exp(-i*kz*z)`; with NumPy's forward FFT the physical outgoing transverse wavevectors are **negative FFT frequencies**. The helper accounts for that sign before attaching each pixel to its Ewald sphere. The 3D multislice scan uses the `volume_n` grid and `volume_angles_deg`. All multislice panels and scans are sampled on the configured physical detector; points beyond the source FFT bandwidth are left undefined.


In [ ]:
cases = ex.multislice_figure(OUT, config=cfg)
show('fig04_multislice_tilts')
if cfg.include_cobalt:
    show('fig04b_cobalt_tilts')
show('fig04c_multislice_volume')

## 5. Check geometry, numerical convergence, and absorption

The checks below independently test the elastic sphere, the rotation, the exact Bragg condition, unit magnetization, and coverage normalization. A selected Bragg-region pattern is recomputed at 1 nm slices. At normal incidence, we compare the integrated $+G$ region against `propagate=False`, which retains local transmission but suppresses inter-slice diffraction.

Do not use convergence at one angle as a universal error bound. For publication, refine lateral sampling, field of view, slice thickness, and angle step independently. Increasing thickness can reduce longitudinal width while absorption lowers the transmitted signal. A geometrically dark rod is not the same as a photon-limited measurement.

Try changing `a`, `energy` and `thickness` together using the formulas above; changing only an image's display scale cannot test the off-Bragg condition. The weak medium is the clean reference for this test. `skyrmion_multislice(..., material='Co')` tests the chosen cobalt response.


In [ ]:
report = ex.validation(OUT, config=cfg)
report

## What to report

Report the energy, tube diameter, lattice spacing, thickness, tilt convention, polarization channel, reciprocal units, shared normalization and sampling coverage. Separate Born-reference plots from production multislice results. Cite the simulator commit and database hashes recorded in `provenance.json`. The accompanying manuscript describes current capabilities and explicitly separates demonstrated results from remaining publication validation.

## Compare with a direct 3D FFT of the skyrmion image

Here we explicitly voxelize the **same** Gaussian-weighted contrast $\mathbf m-(0,0,1)$ inside the slab and call `scipy.fft.fftn` on each 3D component. The Fourier amplitudes include the voxel volume $dx\,dy\,dz$; coordinates are rad/nm. The padded z box contains vacuum on both sides. Increase `fft_nz` at fixed `fft_dz_nm` to refine reciprocal-space interpolation; reduce `fft_dz_nm` to refine the actual z voxels. Arrays in the saved reciprocal volume are ordered `(qx, qy, qz)`.

The comparison shows:

1. **Full 3D FFT** $|F[m_z-1]|^2$: the complete scalar reference, including unmeasured momentum space.
2. **Diffraction assembly**: the Born diffraction patterns placed on their rotated Ewald spheres.
3. **Matched FFT assembly**: complex vector FFTs sampled at exactly the same Ewald coordinates, projected onto the incident direction **before squaring**, and gridded with the same coverage normalization.

For `contrast_channel='xmcd'`, panel 1 is an intrinsic scalar reference; panels 2 and 3 are the physically matched comparison. For `contrast_channel='mz'`, the projection is fixed and all panels represent the same scalar object. Unmeasured voxels stay NaN, and all three panels share one scale. The residual and rod profile quantify agreement. Differences between the analytic thickness envelope and finite z-voxel FFT/interpolation are expected and can be converged.

If a multislice volume generated with **exactly the same configuration** exists, a further panel compares it to the FFT sampled at the multislice scan angles. One globally fitted intensity scale is reported explicitly. This is a shape comparison: multislice includes propagation, and its Gaussian illumination is fixed in the lab, whereas the FFT reference is illuminated in the sample frame. Large tilts, absorption or stronger interaction need not match a Born object. No phase retrieval or real-space 3D reconstruction is implied.


In [ ]:
fft_report = ex.fft_volume_comparison(OUT, config=cfg)
show('fig09_fft_volume_comparison')
show('fig09b_fft_volume_3d')
if fft_report['multislice_comparison'] == 'matched configured scan':
    show('fig09c_multislice_fft_comparison')
print(fft_report)